# Train Best Models (All Tickers × 20 Configurations)

Dieses Notebook lädt für jede Konfiguration (Modeltyp × Horizon × Feature-Set) die besten Hyperparameter aus Ihrer Optuna-DB und trainiert anschließend pro Aktie ein Modell.
Die Modelle werden unter `../data/best_models/{TICKER}/...` gespeichert.

**Konfigurationen (20 je Aktie)**
- Horizonte: `1, 3, 5, 10, 15`
- Feature-Sets: `F1 = ["Close"]`, `F5 = ["Close","pos_mean","neut_mean","neg_mean","news_count"]`
- Modelle: `LSTM`, `xLSTM`

**Wichtig**
- CSV-Pfade: `../data/ts_with_sentiment/{TICKER}_ts_wiht_sentiment.csv` (ohne `_old`)
- Reproduzierbarkeit: `GLOBAL_SEED = 42`


In [2]:
# =========================
# Imports
# =========================
import os
import json
from pathlib import Path
from typing import Dict, Any, List, Literal, Tuple
import shutil

import numpy as np
import torch
import optuna
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

import matplotlib.pyplot as plt
import pandas as pd

from src.model_wrapper import Model, set_global_seed
from src.data_prep import PrepAndDataLoader

optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Info] Using device: {DEVICE}")


[Info] Using device: cuda


In [3]:
# =========================
# Global settings (edit here)
# =========================

# --- Universe ---
TICKERS: List[str] = ["AAPL", "DIS", "IBM", "INTC", "JNJ", "JPM", "KO", "MSFT", "NKE", "V"]

# --- Data settings ---
DATA_DIR = Path("../data/ts_with_sentiment")
FILE_PATTERN = "{ticker}_ts_with_sentiment.csv"

TARGET_COL: str = "Close"
TRAIN_SPLIT: float = 0.6
VAL_SPLIT: float = 0.2

# Feature sets
FEATURE_SETS: Dict[str, List[str]] = {
    "F1": ["Close"],
    "F5": ["Close", "pos_mean", "neut_mean", "neg_mean", "news_count"],
}

# --- Window / horizon settings ---
WINDOW_SIZE: int = 60
HORIZONS: List[int] = [1, 3, 5, 10, 15]
STRIDE: int = 1

# --- Normalization settings ---
NORMALISE: bool = True
NORM_METHOD: Literal["percentage", "minmax"] = "percentage"

# --- Training settings ---
EPOCHS: int = 100
BATCH_SIZE: int = 128
PATIENCE: int = 10
VERBOSE_TRAIN: int = 1

# --- Models ---
MODEL_TYPES: List[str] = ["LSTM", "xLSTM"]

# --- Optuna DB ---
DB_PATH = Path("../data/optuna_studies/BachelorThesis.db")
STORAGE_URL = f"sqlite:///{DB_PATH.as_posix()}"

# --- Output ---
OUT_ROOT = Path("../data/best_models")

# --- Seed ---
GLOBAL_SEED: int = 42

print("[Info] Settings loaded.")
print(f"  Tickers: {TICKERS}")
print(f"  Horizons: {HORIZONS}")
print(f"  Feature sets: {list(FEATURE_SETS.keys())}")
print(f"  Model types: {MODEL_TYPES}")
print(f"  Optuna DB: {DB_PATH}")
print(f"  Output root: {OUT_ROOT}")


[Info] Settings loaded.
  Tickers: ['AAPL', 'DIS', 'IBM', 'INTC', 'JNJ', 'JPM', 'KO', 'MSFT', 'NKE', 'V']
  Horizons: [1, 3, 5, 10, 15]
  Feature sets: ['F1', 'F5']
  Model types: ['LSTM', 'xLSTM']
  Optuna DB: ..\data\optuna_studies\BachelorThesis.db
  Output root: ..\data\best_models


In [4]:
# =========================
# Helpers
# =========================

def ensure_reproducible(seed: int) -> None:
    """Set all relevant seeds and determinism knobs."""
    set_global_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def plot_history_from_json(history_json_path: str, title: str = "") -> None:
    """Plot train/val loss from a JSON history file saved by Model.train."""
    if not os.path.isfile(history_json_path):
        print(f"[WARN] History file not found: {history_json_path}")
        return
    with open(history_json_path, "r") as f:
        hist = json.load(f)

    plt.figure(figsize=(8, 4))
    if "loss" in hist:
        plt.plot(hist["loss"], label="train_loss")
    if "val_loss" in hist:
        plt.plot(hist["val_loss"], label="val_loss")
    plt.title(f"Training History {title}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


def study_name(model_type: str, horizon: int, n_features: int) -> str:
    return f"{model_type}_H{horizon}_F{n_features}"


def load_best_trial_config(study: optuna.study.Study) -> Tuple[Dict[str, Any], optuna.trial.FrozenTrial]:
    """Select a single best trial from a multi-objective Pareto set (RMSE min, then R² max)."""
    pareto_trials = study.best_trials
    if not pareto_trials:
        raise RuntimeError("Study has no Pareto-optimal trials (best_trials empty).")

    best_trial = min(pareto_trials, key=lambda t: (t.values[0], -t.values[1]))
    cfg = best_trial.user_attrs.get("config", None)
    if cfg is None:
        raise RuntimeError("Best trial has no stored 'config' in user_attrs.")
    return cfg, best_trial


def safe_mkdir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def save_json(path: Path, obj: Any) -> None:
    safe_mkdir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Compute RMSE and R² on flattened arrays."""
    y_true_flat = y_true.reshape(-1)
    y_pred_flat = y_pred.reshape(-1)
    rmse = float(root_mean_squared_error(y_true_flat, y_pred_flat))
    r2 = float(r2_score(y_true_flat, y_pred_flat))
    mse = float(mean_squared_error(y_true_flat, y_pred_flat))
    return {"mse": mse, "rmse": rmse, "r2": r2}


In [5]:

def train_one(
    ticker: str,
    model_type: str,
    horizon: int,
    feature_key: str,          # z.B. "F1" oder "F5"
    feature_cols: List[str],
) -> Dict[str, Any]:
    """
    Trainiert ein Modell für (ticker, model_type, horizon, feature_set) mit den besten Optuna-Hyperparametern,
    speichert nur das "beste Modell" mit Namensschema:
        {MODEL}_H{H}_F{F}_{TICKER}.pt
    und speichert zusätzlich Plots + Prediction-Tabellen unter ../data/Figures/(Plots|Tables)/{TICKER}/
    """

    # -------------------------
    # Setup / Paths
    # -------------------------
    ensure_reproducible(GLOBAL_SEED)

    csv_path = DATA_DIR / FILE_PATTERN.format(ticker=ticker)
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing CSV for {ticker}: {csv_path}")

    # naming scheme parts
    n_features_expected = len(feature_cols)
    base_name = f"{model_type}_H{horizon}_F{n_features_expected}_{ticker}"

    # output dirs
    model_dir = OUT_ROOT / ticker
    fig_plot_dir = Path("../data/Figures/Plots") / ticker
    fig_table_dir = Path("../data/Figures/Tables") / ticker
    safe_mkdir(model_dir)
    safe_mkdir(fig_plot_dir)
    safe_mkdir(fig_table_dir)

    # final artifact paths (same basename everywhere)
    final_ckpt_path   = model_dir / f"{base_name}.pt"
    final_cfg_path    = model_dir / f"{base_name}.config.json"
    final_metrics_path= model_dir / f"{base_name}.metrics.json"
    final_hist_path   = model_dir / f"{base_name}.history.json"

    plot_pdf_path     = fig_plot_dir / f"{base_name}.pdf"
    plot_png_path     = fig_plot_dir / f"{base_name}.png"
    table_csv_path    = fig_table_dir / f"{base_name}.csv"
    table_tex_path    = fig_table_dir / f"{base_name}.tex"

    # -------------------------
    # Data prep
    # -------------------------
    prep = PrepAndDataLoader(
        filename=str(csv_path),
        training_split=TRAIN_SPLIT,
        validation_split=VAL_SPLIT,
        cols=feature_cols,
        target_col=TARGET_COL,
    )

    tmp_model = Model()
    X_tr, y_tr, X_va, y_va, X_te, y_te = tmp_model.prepare_data_from_prep(
        prep,
        normalise=NORMALISE,
        window_size=WINDOW_SIZE,
        prediction_range=horizon,
        norm_method=NORM_METHOD,
        stride=STRIDE,
        verbose=0,
    )

    # dates + baseT for denorm
    dates_te = prep.get_prediction_dates(
        "test",
        window_size=WINDOW_SIZE,
        prediction_range=horizon,
        stride=STRIDE,
    )  # [N, H]
    baseT_te = tmp_model._cache.get("baseT_te", None)

    n_features_actual = int(X_tr.shape[-1])
    if n_features_actual != n_features_expected:
        # schützt vor stillen Spalten-/CSV-Problemen
        raise ValueError(
            f"{ticker} {base_name}: expected n_features={n_features_expected}, "
            f"but got {n_features_actual} from prepared data."
        )

    # -------------------------
    # Load best Optuna config
    # -------------------------
    sname = study_name(model_type, horizon, n_features_actual)
    study = optuna.load_study(study_name=sname, storage=STORAGE_URL)
    best_cfg, best_trial = load_best_trial_config(study)

    # enforce task-specific fields (robust against stale configs)
    best_cfg = dict(best_cfg)
    best_cfg["window_size"] = int(WINDOW_SIZE)
    best_cfg["n_features"]  = int(n_features_actual)
    best_cfg["horizon"]     = int(horizon)

    # -------------------------
    # Train (checkpoint temp dir)
    # -------------------------
    tmp_ckpt_dir = (OUT_ROOT / "_tmp_checkpoints" / ticker / base_name)
    safe_mkdir(tmp_ckpt_dir)

    model = Model()
    model.build(config=best_cfg, model_type=model_type)

    best_ckpt_path = model.train(
        X_tr, y_tr,
        X_va, y_va,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        save_dir=str(tmp_ckpt_dir),
        patience=PATIENCE,
        monitor="val_loss",
        verbose=VERBOSE_TRAIN,
        restore_best_weights=True,
        save_best_only=True,
    )

    # copy "best model" to final naming scheme
    shutil.copy2(best_ckpt_path, final_ckpt_path)

    # optional: copy history json (if exists)
    hist_src = str(best_ckpt_path).replace(".pt", "_history.json")
    if os.path.isfile(hist_src):
        try:
            with open(hist_src, "r", encoding="utf-8") as f:
                hist = json.load(f)
            save_json(final_hist_path, hist)
        except Exception as e:
            print(f"[WARN] Could not save history for {base_name}: {e}")

    # -------------------------
    # Predict on TEST (scaled), then denormalise to LEVEL
    # -------------------------
    y_test_scaled = y_te                       # [N, H, 1]
    y_pred_scaled = model.predict_multi_horizon(
        X_te,
        batch_size=BATCH_SIZE,
        verbose=0,
    )  # [N, H]

    # denormalise
    y_pred_scaled_3d = y_pred_scaled[..., None]        # [N, H, 1]
    y_true_scaled_3d = y_test_scaled                    # [N, H, 1]

    if NORMALISE:
        if NORM_METHOD == "percentage":
            if baseT_te is None:
                raise ValueError(f"{base_name}: baseT_te is required for percentage normalisation.")
            y_pred_level = prep.denormalise(
                y_pred_scaled_3d, method="percentage", base_values=baseT_te, normalise=True
            ).squeeze(-1)  # [N, H]
            y_true_level = prep.denormalise(
                y_true_scaled_3d, method="percentage", base_values=baseT_te, normalise=True
            ).squeeze(-1)
        elif NORM_METHOD == "minmax":
            y_pred_level = prep.denormalise(
                y_pred_scaled_3d, method="minmax", base_values=None, normalise=True
            ).squeeze(-1)
            y_true_level = prep.denormalise(
                y_true_scaled_3d, method="minmax", base_values=None, normalise=True
            ).squeeze(-1)
        else:
            raise ValueError(f"{base_name}: unknown NORM_METHOD={NORM_METHOD}")
    else:
        y_pred_level = y_pred_scaled
        y_true_level = y_test_scaled.squeeze(-1)

    # metrics (LEVEL)
    y_true_flat = y_true_level.reshape(-1)
    y_pred_flat = y_pred_level.reshape(-1)
    test_rmse_level = float(root_mean_squared_error(y_true_flat, y_pred_flat))
    test_r2_level   = float(r2_score(y_true_flat, y_pred_flat))

    # -------------------------
    # Save prediction "table" for LaTeX (last horizon t+H)
    # -------------------------
    if STRIDE == 1:
        h_idx = horizon - 1
        df_pred = pd.DataFrame({
            "date": dates_te[:, h_idx],
            "true": y_true_level[:, h_idx],
            "pred": y_pred_level[:, h_idx],
        }).sort_values("date").reset_index(drop=True)
    else:
        # general fallback: merged dates
        df_pred = pd.DataFrame({
            "date": dates_te.reshape(-1),
            "true": y_true_level.reshape(-1),
            "pred": y_pred_level.reshape(-1),
        }).groupby("date", as_index=False).mean().sort_values("date").reset_index(drop=True)

    df_pred.to_csv(table_csv_path, index=False)
    df_pred.to_latex(
        table_tex_path,
        index=False,
        float_format="%.6f",
        longtable=True,
        caption=f"{ticker} – {model_type}, H={horizon}, F={n_features_actual} (Test, level)",
        label=f"tab:{base_name}".replace("-", "_"),
    )

    # -------------------------
    # Save plot (PDF + PNG) for LaTeX
    # -------------------------
    plt.figure(figsize=(12, 5))

    if STRIDE == 1:
        h_idx = horizon - 1
        plt.plot(dates_te[:, h_idx], y_true_level[:, h_idx], label=f"True (t+{horizon})")
        plt.plot(dates_te[:, h_idx], y_pred_level[:, h_idx], label=f"Pred (t+{horizon})")
    else:
        plt.plot(df_pred["date"], df_pred["true"], label="True (merged)")
        plt.plot(df_pred["date"], df_pred["pred"], label="Pred (merged)")

    plt.title(
        f"{ticker} – {model_type} | H={horizon} | F={n_features_actual} ({feature_key}) – TEST\n"
        f"RMSE={test_rmse_level:.4f}, R²={test_r2_level:.3f}"
    )
    plt.xlabel("Date (Test set)")
    plt.ylabel(TARGET_COL)
    plt.legend()
    plt.grid(True)
    plt.gcf().autofmt_xdate()
    plt.tight_layout()

    plt.savefig(plot_pdf_path, format="pdf", bbox_inches="tight")
    plt.savefig(plot_png_path, format="png", dpi=300, bbox_inches="tight")
    plt.close()

    # -------------------------
    # Persist config + metrics (same basename)
    # -------------------------
    save_json(final_cfg_path, best_cfg)
    save_json(final_metrics_path, {
        "ticker": ticker,
        "model_type": model_type,
        "horizon": horizon,
        "feature_key": feature_key,
        "feature_cols": feature_cols,
        "n_features": n_features_actual,
        "study_name": sname,
        "optuna_best_trial": {
            "trial_number": int(best_trial.number),
            "values": [float(best_trial.values[0]), float(best_trial.values[1])],
            "params": dict(best_trial.params),
        },
        "test_level": {
            "rmse": test_rmse_level,
            "r2": test_r2_level,
        },
        "artifacts": {
            "checkpoint": str(final_ckpt_path),
            "config": str(final_cfg_path),
            "metrics": str(final_metrics_path),
            "history": str(final_hist_path) if final_hist_path.is_file() else None,
            "plot_pdf": str(plot_pdf_path),
            "plot_png": str(plot_png_path),
            "table_csv": str(table_csv_path),
            "table_tex": str(table_tex_path),
        }
    })

    return {
        "ticker": ticker,
        "model_type": model_type,
        "horizon": horizon,
        "feature_key": feature_key,
        "n_features": n_features_actual,
        "checkpoint": str(final_ckpt_path),
        "plot_pdf": str(plot_pdf_path),
        "table_tex": str(table_tex_path),
        "test_rmse_level": test_rmse_level,
        "test_r2_level": test_r2_level,
    }


In [6]:
# =========================
# Runner: all tickers × 20 configurations
# =========================

ensure_reproducible(GLOBAL_SEED)

results: List[Dict[str, Any]] = []
failures: List[Dict[str, Any]] = []

total_jobs = len(TICKERS) * len(MODEL_TYPES) * len(HORIZONS) * len(FEATURE_SETS)
job_i = 0

for ticker in TICKERS:
    for model_type in MODEL_TYPES:
        for horizon in HORIZONS:
            for feature_key, feature_cols in FEATURE_SETS.items():
                job_i += 1
                tag = f"{ticker} | {model_type} | H{horizon} | {feature_key}"
                print(f"\n[{job_i}/{total_jobs}] Training: {tag}")

                try:
                    summary = train_one(
                        ticker=ticker,
                        model_type=model_type,
                        horizon=horizon,
                        feature_key=feature_key,
                        feature_cols=feature_cols,
                    )
                    results.append(summary)
                    print(f"[OK] Saved to: {summary['run_dir']}")
                    print(f"     Val:  RMSE={summary['val_rmse']:.6f} | R²={summary['val_r2']:.4f}")
                    print(f"     Test: RMSE={summary['test_rmse']:.6f} | R²={summary['test_r2']:.4f}")

                except Exception as e:
                    print(f"[FAIL] {tag}: {e}")
                    failures.append({
                        "ticker": ticker,
                        "model_type": model_type,
                        "horizon": horizon,
                        "feature_key": feature_key,
                        "error": str(e),
                    })

print("\n=========================")
print("[Done] Training loop finished.")
print(f"  Successful runs: {len(results)}")
print(f"  Failures:        {len(failures)}")

# Persist a global summary
safe_mkdir(OUT_ROOT)
save_json(OUT_ROOT / "training_summary_results.json", results)
save_json(OUT_ROOT / "training_summary_failures.json", failures)

df_results = pd.DataFrame(results).sort_values(["ticker", "model_type", "horizon", "n_features"])
df_results



[1/200] Training: AAPL | LSTM | H1 | F1
[Model] LSTM compiled:
  Input: seq_len=60, features=1 | hidden=64 | horizon=1
  LSTM layers=2
  Trainable params: 50,497
[Model] Training start | epochs=100 | batch_size=128 | horizon=1
Epoch 001/100 - loss=0.018459 - val_loss=0.026358
[Checkpoint] Saved best model to: ..\data\best_models\_tmp_checkpoints\AAPL\LSTM_H1_F1_AAPL\LSTM_H1_20251218-144944_b128_un64_lay2_e100.pt (val_loss=0.026358)
Epoch 002/100 - loss=0.010644 - val_loss=0.007939
[Checkpoint] Saved best model to: ..\data\best_models\_tmp_checkpoints\AAPL\LSTM_H1_F1_AAPL\LSTM_H1_20251218-144944_b128_un64_lay2_e100.pt (val_loss=0.007939)
Epoch 003/100 - loss=0.002748 - val_loss=0.002105
[Checkpoint] Saved best model to: ..\data\best_models\_tmp_checkpoints\AAPL\LSTM_H1_F1_AAPL\LSTM_H1_20251218-144944_b128_un64_lay2_e100.pt (val_loss=0.002105)
Epoch 004/100 - loss=0.001356 - val_loss=0.001881
[Checkpoint] Saved best model to: ..\data\best_models\_tmp_checkpoints\AAPL\LSTM_H1_F1_AAPL\LST

,ticker,model_type,horizon,feature_key,n_features,checkpoint,plot_pdf,table_tex,test_rmse_level,test_r2_level
0,AAPL,LSTM,1,F1,1,..\data\best_models\AAPL\LSTM_H1_F1_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H1_F1_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H1_F1_AAPL.tex,2.931247,0.991299
1,AAPL,LSTM,1,F5,5,..\data\best_models\AAPL\LSTM_H1_F5_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H1_F5_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H1_F5_AAPL.tex,2.987633,0.990961
2,AAPL,LSTM,3,F1,1,..\data\best_models\AAPL\LSTM_H3_F1_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H3_F1_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H3_F1_AAPL.tex,4.205807,0.982047
3,AAPL,LSTM,3,F5,5,..\data\best_models\AAPL\LSTM_H3_F5_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H3_F5_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H3_F5_AAPL.tex,4.679686,0.977774
4,AAPL,LSTM,5,F1,1,..\data\best_models\AAPL\LSTM_H5_F1_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H5_F1_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H5_F1_AAPL.tex,5.092688,0.973614
...,...,...,...,...,...,...,...,...,...,...
195,V,xLSTM,5,F5,5,..\data\best_models\V\xLSTM_H5_F5_V.pt,..\data\Figures\Plots\V\xLSTM_H5_F5_V.pdf,..\data\Figures\Tables\V\xLSTM_H5_F5_V.tex,6.596665,0.966921
196,V,xLSTM,10,F1,1,..\data\best_models\V\xLSTM_H10_F1_V.pt,..\data\Figures\Plots\V\xLSTM_H10_F1_V.pdf,..\data\Figures\Tables\V\xLSTM_H10_F1_V.tex,6.713783,0.965017
197,V,xLSTM,10,F5,5,..\data\best_models\V\xLSTM_H10_F5_V.pt,..\data\Figures\Plots\V\xLSTM_H10_F5_V.pdf,..\data\Figures\Tables\V\xLSTM_H10_F5_V.tex,7.925166,0.951254
198,V,xLSTM,15,F1,1,..\data\best_models\V\xLSTM_H15_F1_V.pt,..\data\Figures\Plots\V\xLSTM_H15_F1_V.pdf,..\data\Figures\Tables\V\xLSTM_H15_F1_V.tex,7.851080,0.951417


## Optional: Schnellvergleich der Ergebnisse

Die folgenden Zellen sind optional und dienen nur der Übersicht (kein Training).  
Sie lesen die im Lauf erzeugte `training_summary_results.json` und zeigen einfache Aggregationen.


In [7]:
# =========================
# Optional: load summary + basic pivots
# =========================

summary_path = OUT_ROOT / "training_summary_results.json"
if summary_path.is_file():
    with open(summary_path, "r", encoding="utf-8") as f:
        results_loaded = json.load(f)
    df = pd.DataFrame(results_loaded)

    display(df.head(10))

    # Pivot: mean test RMSE per ticker / model_type / horizon / feature_key
    pivot_rmse = df.pivot_table(
        index=["ticker"],
        columns=["model_type", "horizon", "feature_key"],
        values="test_rmse",
        aggfunc="mean",
    )
    display(pivot_rmse)

    # Pivot: mean test R2
    pivot_r2 = df.pivot_table(
        index=["ticker"],
        columns=["model_type", "horizon", "feature_key"],
        values="test_r2",
        aggfunc="mean",
    )
    display(pivot_r2)

else:
    print(f"[WARN] Summary not found: {summary_path}")


,ticker,model_type,horizon,feature_key,n_features,checkpoint,plot_pdf,table_tex,test_rmse_level,test_r2_level
0,AAPL,LSTM,1,F1,1,..\data\best_models\AAPL\LSTM_H1_F1_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H1_F1_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H1_F1_AAPL.tex,2.931247,0.991299
1,AAPL,LSTM,1,F5,5,..\data\best_models\AAPL\LSTM_H1_F5_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H1_F5_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H1_F5_AAPL.tex,2.987633,0.990961
2,AAPL,LSTM,3,F1,1,..\data\best_models\AAPL\LSTM_H3_F1_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H3_F1_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H3_F1_AAPL.tex,4.205807,0.982047
3,AAPL,LSTM,3,F5,5,..\data\best_models\AAPL\LSTM_H3_F5_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H3_F5_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H3_F5_AAPL.tex,4.679686,0.977774
4,AAPL,LSTM,5,F1,1,..\data\best_models\AAPL\LSTM_H5_F1_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H5_F1_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H5_F1_AAPL.tex,5.092688,0.973614
5,AAPL,LSTM,5,F5,5,..\data\best_models\AAPL\LSTM_H5_F5_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H5_F5_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H5_F5_AAPL.tex,5.501644,0.969206
6,AAPL,LSTM,10,F1,1,..\data\best_models\AAPL\LSTM_H10_F1_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H10_F1_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H10_F1_AAPL.tex,6.980505,0.950283
7,AAPL,LSTM,10,F5,5,..\data\best_models\AAPL\LSTM_H10_F5_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H10_F5_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H10_F5_AAPL.tex,8.042494,0.934005
8,AAPL,LSTM,15,F1,1,..\data\best_models\AAPL\LSTM_H15_F1_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H15_F1_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H15_F1_AAPL.tex,9.086840,0.915563
9,AAPL,LSTM,15,F5,5,..\data\best_models\AAPL\LSTM_H15_F5_AAPL.pt,..\data\Figures\Plots\AAPL\LSTM_H15_F5_AAPL.pdf,..\data\Figures\Tables\AAPL\LSTM_H15_F5_AAPL.tex,9.726565,0.903255


KeyError: 'test_rmse'